# Monty.jl SSA Simulation
For most simulations, only modify parameters contained within #-----#s


# Initialization
Run this FIRST, and only ONCE, when starting a new project or when you're in a new folder. This creates the environment with the necessary packages to function.

In [ ]:
using Pkg
Pkg.activate(".")
Pkg.add(url="https://github.com/LithosCarbon/Monty.jl")
Pkg.add([
    "Random",
    "Distributions",
    "Meshes",
    "GeoStatsFunctions",
    "GeoStatsProcesses",
    "DataFrames",
    "Statistics",
    "Plots",
    "DimensionalData",
    "CairoMakie",
    "CSV"
])

# 1. Deployment and Feedstock Details
This section is information for the setup of field and details about the feedstock. Change parameters as necessary to cohere with your setup. Units are listed in parentheses. 

Ignore everything above Sample_Name.

In [ ]:
using Pkg
Pkg.activate(".")

using Monty, Random, Distributions, Meshes
using GeoStatsFunctions, GeoStatsProcesses
using DataFrames, Statistics
using Plots
using DimensionalData
const rng = Xoshiro(42)
#---------------------------------------------------------------------------------#
Sample_Name = "TARIY1" # Change Wemby to the feedstock name.
A = 1000 # Area of treatment (m^2)
A_var = 50 # Variation in treatment area (m^2)
Q_μ = 2.0 # Application rate (kg/m^2)
d = 0.15 # Sample core depth (m)
d_var = 0.025 # Sample depth variation (m)
p_soil = 1.36e3 # Soil Density (kg/m^3)
p_soil_var = 50 # Soil Variation (kg/m^3)
phi = 0.125 # Feedstock moisture fraction 
#---------------------------------------------------------------------------------#

Mtot = Q_μ * A # Total mass applied, Q*A. You do not need to change anything, this is autocalculated.

# 2. Rock Composition Details
This section is information for the composition of the feedstock. Change parameters as necessary to cohere with your setup.

In [ ]:
#---------------------------------------------------------------------------------#
# Define tracked mineral names (Add/remove using symbols starting with a colon)
minerals = (:Plagioclase, 
            :Clinopyroxene, 
            :Quartz, 
            :Sericite, 
            :FeTiOxides, 
            :Chlorite, 
            :Actinolite, 
            :Olivine, 
            :Orthoclase, 
            :Nepheline, 
            :Muscovite, 
            :Apatite,
            :Calcite
            )
n_minerals = length(minerals) 

pH = 4.0
infiltration = 0.06553079948
SA = 10 # Surface Area (m^2/g)

# Elemental proportions for each mineral 
# Brackets are in the following order: [Ca, Mg, Na, K] 
# (Each row must match the order above)
m_element_arr = [
    [7.8/100, 0, 3.5/100, 0.4/100],   # Plagioclase
    [5.3/100, 8.8/100, 0, 0],         # Clinopyroxene
    [0.0, 0.0, 0.0, 0.0],             # Quartz
    [0.0, 0.0, 1.3/100, 9.81/100],    # Sericite
    [1.12/100, 0.0, 0.0, 0.0],        # FeTiOxides
    [0.0, 9/100, 0.0, 0.0],           # Chlorite
    [10.4/100, 8.13/100, 0.0, 0.0],   # Actinolite
    [0, 34.55/100, 0, 0],             # Olivine
    [0, 0, 0, 14.05/100],             # Orthoclase
    [0, 0, 11.80/100, 6.69/100],      # Nepheline
    [0, 0, 0, 9.81/100],              # Muscovite
    [39.36/100, 0, 0, 0],             # Apatite
    [40.04/100, 0, 0, 0]              # Calcite
]

# Relative mineral abundances in feedstock (these MUST sum to ~1.0)
m_f_arr = [
    0.588,  # Plagioclase
    0.178,  # Clinopyroxene
    0.0,    # Quartz
    0.0,    # Sericite
    0.027,  # FeTiOxides
    0.023,  # Chlorite
    0.035,  # Actinolite
    0.0,    # Olivine
    0.126,  # Orthoclase
    0.014,  # Nepheline
    0.019,  # Muscovite
    0.0,    # Apatite
    0.0     # Calcite
]

f_var = 0.05 # Variation multiplier for feedstock mineralogy

# Baseline background concentrations of each mineral in the soil prior to application
m_s_arr = [
    Normal(1.0e-10,1.0e-10/10),  # Plagioclase
    Normal(1.0e-10,1.0e-10/10),  # Clinopyroxene
    Normal(1.0e-10,1.0e-10/10),  # Quartz
    Normal(1.0e-10,1.0e-10/10),  # Sericite
    Normal(1.0e-10,1.0e-10/10),  # FeTiOxides
    Normal(1.0e-10,1.0e-10/10),  # Chlorite
    Normal(1.0e-10,1.0e-10/10),  # Actinolite
    Normal(1.0e-10,1.0e-10/10),  # Olivine
    Normal(1.0e-10,1.0e-10/10),  # Orthoclase
    Normal(1.0e-10,1.0e-10/10),  # Nepheline
    Normal(1.0e-10,1.0e-10/10),  # Muscovite
    Normal(1.0e-10,1.0e-10/10),  # Apatite
    Normal(1.0e-10,1.0e-10/10)   # Calcite
]

# change the value of SR (saturation ratio), 
# k_acid (acid coefficient exp), 
# k_neutral (neutral coefficient exp), 
# n (acid exponential), and 
# mm (molar mass in g/mol) in accordance with the properties of the mineral.
mineral_props = Dict( 

    :Plagioclase   => Dict("SR"=>0.778, "k_acid"=>-7.87,  "k_neutral"=>-10.91, "n"=>0.626, "mm"=>271.81),
    :Clinopyroxene => Dict("SR"=>0.555, "k_acid"=>-6.82,  "k_neutral"=>-11.97, "n"=>0.7,   "mm"=>236.35),
    :Quartz        => Dict("SR"=>0.5,   "k_acid"=>-15.0,  "k_neutral"=>-15.0,  "n"=>0.5,   "mm"=>60.08),
    :Sericite      => Dict("SR"=>0.47,  "k_acid"=>-11.85, "k_neutral"=>-13.85, "n"=>0.37,  "mm"=>398.71),
    :FeTiOxides    => Dict("SR"=>0.5,   "k_acid"=>-15.0,  "k_neutral"=>-15.0,  "n"=>0.5,   "mm"=>100.0),
    :Chlorite      => Dict("SR"=>0.33,  "k_acid"=>-11.11, "k_neutral"=>-12.52, "n"=>0.5,   "mm"=>595.22),
    :Actinolite    => Dict("SR"=>0.558, "k_acid"=>-8.4,   "k_neutral"=>-10.6,  "n"=>0.7,   "mm"=>853.16),
    :Olivine       => Dict("SR"=>0.437, "k_acid"=>-4.8,   "k_neutral"=>0,      "n"=>1.0,   "mm"=>140.69),
    :Orthoclase    => Dict("SR"=>0.604, "k_acid"=>-10.06, "k_neutral"=>-12.41, "n"=>0.5,   "mm"=>278.33),
    :Nepheline     => Dict("SR"=>0.451, "k_acid"=>-2.73,  "k_neutral"=>-8.56,  "n"=>1.13,  "mm"=>146.08),
    :Muscovite     => Dict("SR"=>0.470, "k_acid"=>-11.85, "k_neutral"=>-13.55, "n"=>0.37,  "mm"=>398.71),
    :Apatite       => Dict("SR"=>0.592, "k_acid"=>-5.63,  "k_neutral"=>0,      "n"=>0.8,   "mm"=>509.12),
    :Calcite       => Dict("SR"=>0.634, "k_acid"=>-2.05,  "k_neutral"=>0,      "n"=>1.0,   "mm"=>100.09)
)
#---------------------------------------------------------------------------------#

# Lambda Calculation Function
function calculate_lambda(SA, k_acid, k_neutral, pH, n, SR, mm, infilt)
    rate = (10^(k_acid - pH * n) + 10^k_neutral)
    dmdt_yr = rate * (1 - SR) * SA * infilt * (3600 * 24 * 365) * mm
    dmdt_yr = min(dmdt_yr, 0.9999) 
    return -log(1 - dmdt_yr)
end

# Dynamic Rate Calculation
leach_arr = [calculate_lambda(SA, mineral_props[m]["k_acid"], mineral_props[m]["k_neutral"], pH, 
             mineral_props[m]["n"], mineral_props[m]["SR"], mineral_props[m]["mm"], infiltration) for m in minerals]

leach_arr

# 3. Treatment Area and Sampling
This section determines the parameters of the application and control areas, as well as the temporality and locations of the sampling rounds.

In [ ]:
#---------------------------------------------------------------------------------#
# Define temporal sampling schedule (in years)
rounds = [-.03, 0.05, 1.0, 2.0, 5.0, 15.0]

# Construct grid framework (n_x by n_y grid of width x height cells)
n_x = 4 
n_y = 5 
width = 10.0 # in meters
height = 10.0 # in meters
grid = CartesianGrid((n_x, n_y), (0.0, 0.0), (width, height))
#---------------------------------------------------------------------------------#

# 4. Simulation! and Results
Self-explanatory. Basic simulations do not require any changes to this section.

In [ ]:
# Total number of iterations
nsim = 1000

# Segment grid into spatial zones (Left half = Treatment, Right half = Control) 
treatment = reduce(vcat, [grid[i, :] for i ∈ 1:Int64(n_x/2)])
control = reduce(vcat, [grid[i, :] for i ∈ (Int64(n_x/2 + 1)):Int64(n_x)])

# Layout noise (jittering)
plannedjitter = GridCentroidJitter(width/2, 2.0) 
treatment_jitter = plannedjitter.(treatment .|> centroid)
control_jitter = plannedjitter.(control .|> centroid)

# Sampling plan
plan = pairedsampleplan(treatment_jitter, control_jitter, rounds)
# Composite core framework
samp = CoreSet(plan, 5)

# Initialize physical simulation constants
sim = Simulation(minerals, samp)
sim.ρf .= 2.98e3 # Constant feedstock density (g/cm^3 scaling equivalent)

Q = GaussianSimulator(samp, Q_μ)
Q_cov = GaussianCovariance(MetricBall((10.0, 100.0)), nugget=(3/10)^2, sill=(1/3)^2)
stencil = CircleStencil(ncore(sim), 1.0)
samplerjitter = Jitter(1.0)
corejitter = Jitter(0.1)

cs = NamedTuple{minerals}(Tuple(m_s_arr))
depth = TriangularDist(d - d_var, d + d_var)
ρs = Normal(p_soil, p_soil_var)
feed_dist = [Normal(m, f_var * m) for m in m_f_arr]
cf = NamedTuple{minerals}(Tuple(feed_dist))
leach_dist = [ExponentialLeaching(λ=λ) for λ in leach_arr]
leach = NamedTuple{minerals}(Tuple(leach_dist))
noise_arr = fill(0.03, n_minerals)

sims = simulationstack(nsim, sim, samp, plan) do
    executeplan!(samp, plan, stencil=stencil, plannedjitter=plannedjitter, samplerjitter=samplerjitter, corejitter=corejitter)
    updategaussian!(Q, samp.points, Q_cov)
    spreading!(rng, sim, plan, Q)
    clamp!(sim.Q, 0.0, Inf)
    unmixed!(rng, sim, depth=depth)
    rand!(rng, ρs, sim.ρs)

    for analyte ∈ minerals
        rand!(rng, cf[analyte], sim.cf[analyte])
        rand!(rng, cs[analyte], sim.cs[analyte])
        leaching!(sim, analyte, leach[analyte], plan.time)
    end

    massloss!(sim, plan, mean)
    analyze!(rng, sim, NamedTuple{minerals}(Tuple(noise_arr)), 0.005)
end

resultdir = joinpath(@__DIR__, "Results", string(Sample_Name, "_simulation"))
csvdir = joinpath(resultdir, "csv")
mkpath(csvdir)

# Export datasets
foreach(i -> tocsv(joinpath(csvdir, string(Sample_Name, "_$i.csv")), sims, i), 1:nsim)
println("Simulation complete! Data exported to: ", csvdir)

# Graphs, Charts, and Figures


Spreading Pattern

In [ ]:
import CairoMakie as Mke

plotdir = joinpath(resultdir, "plots")
mkpath(plotdir)

# spreading realization
fig1 = Mke.Figure(size=(800, 300))
densegrid = CartesianGrid(minimum(grid), maximum(grid), dims=(150, 150))
γ = Q_cov.γ
r = rand(rng, γ |> GaussianProcess, densegrid, :z => Float64, 6)

for j ∈ 1:3
    i = 1
    ax = Mke.Axis(
        fig1[i, j],
        title="Realized Spreading Pattern $(3*(i-1)+j)",
        xlabel="Easting [m]",
        ylabel="Northing [m]",
        aspect=1,
    )
    z = clamp.(r[i+j].z .+ Q_μ, 0.0, Inf)
    viz!(ax, r[i+j].geometry, color=z)
    viz!(ax, control, color=:gray)
end

Mke.Colorbar(
    fig1[:, 4],
    limits=(
        clamp(minimum(member -> minimum(member.z) + Q_μ, r), 0.0, Inf),
        maximum(member -> maximum(member.z) + Q_μ, r),
    ),
    colormap=:viridis,
    label="Applied Feedstock [kg/m^2]",
)

Mke.save(joinpath(plotdir, "spreading_realizations.png"), fig1)
fig1

# Ignore 'ERROR: LoadError: UndefVarError: `expand_dimensions` not defined' for now--not sure why it's happening but it doesn't affect anything


In [ ]:
x = map(p -> p.coords[1], samp.points |> vec)
y = map(p -> p.coords[2], samp.points |> vec)
size = 5

Application Rate [kg/m^2]

In [ ]:
fig_app = Mke.Figure(size=(600, 450))

ax1 = Mke.Axis(
    fig_app[1, 1],
    aspect=1,
    xlabel="Easting [m]",
    ylabel="Northing [m]",
    title="Application Rate [kg/m^2]",
)
viz!(ax1, grid, color=:white, showfacets=true)
s1 = Mke.scatter!(ax1, x, y, color=sim.Q |> vec, markersize=size)
Mke.Colorbar(fig_app[1, 2], s1)

Mke.save(joinpath(plotdir, "application_rate.png"), fig_app)
fig_app

Soil Bulk Density [kg/m^3]

In [ ]:
fig_rho = Mke.Figure(size=(600, 450))

ax2 = Mke.Axis(
    fig_rho[1, 1],
    aspect=1,
    xlabel="Easting [m]",
    ylabel="Northing [m]",
    title="Soil Bulk Density [kg/m^3]",
)
viz!(ax2, grid, color=:white, showfacets=true)
s2 = Mke.scatter!(
    ax2,
    x,
    y,
    color=sim.ρs |> vec,
    colormap=:bamako,
    markersize=size,
)
Mke.Colorbar(fig_rho[1, 2], s2)

Mke.save(joinpath(plotdir, "soil_bulk_density.png"), fig_rho)
fig_rho

Sampling Round(s)

In [ ]:
fig_rounds = Mke.Figure(size=(500, 450))
n_rounds = length(rounds)

ax3 = Mke.Axis(
    fig_rounds[1, 1],
    aspect=1,
    xlabel="Easting [m]",
    ylabel="Northing [m]",
    title="Sampling Round (1 - $n_rounds)",
)
viz!(ax3, grid, color=:white, showfacets=true)
Mke.scatter!(
    ax3,
    x,
    y,
    color=repeat(plan.round, 1, 5) |> transpose |> vec,
    colormap=:tab10,
    colorrange=(1, 10),
    markersize=size,
)

Mke.save(joinpath(plotdir, "sampling_rounds.png"), fig_rounds)
fig_rounds

Core Depth [cm]

In [ ]:
fig_depth = Mke.Figure(size=(600, 450))

ax4 = Mke.Axis(
    fig_depth[1, 1],
    aspect=1,
    xlabel="Easting [m]",
    ylabel="Northing [m]",
    title="Core Depth [cm]",
)
viz!(ax4, grid, color=:white, showfacets=true)
s4 = Mke.scatter!(
    ax4,
    x,
    y,
    color=100 * sim.d |> vec,
    colormap=:reds,
    markersize=size,
)
Mke.Colorbar(fig_depth[1, 2], s4)

Mke.save(joinpath(plotdir, "core_depth.png"), fig_depth)
fig_depth

Soil Mineral X [ppm]

In [ ]:
fig_plag = Mke.Figure(size=(600, 450))

ax5 = Mke.Axis(
    fig_plag[1, 1],
    aspect=1,
    xlabel="Easting [m]",
    ylabel="Northing [m]",
    title="Soil Plagioclase [ppm]",
)
viz!(ax5, grid, color=:white, showfacets=true)
s5 = Mke.scatter!(
    ax5,
    x,
    y,
    color=1e6 * sim.cs[:Plagioclase] |> vec,
    colormap=:turbo,
    markersize=size,
)
Mke.Colorbar(fig_plag[1, 2], s5)

Mke.save(joinpath(plotdir, "soil_plagioclase.png"), fig_plag)
fig_plag

Soil Element X Concentration [ppm]

In [ ]:
# Mg is the 2nd element in the [Ca, Mg, Na, K] tracking array
mg_concentration = sum(m_element_arr[i][2] .* sim.cs[minerals[i]] for i in 1:length(minerals))

fig_mg = Mke.Figure(size=(600, 450))

ax_mg = Mke.Axis(
    fig_mg[1, 1],
    aspect=1,
    xlabel="Easting [m]",
    ylabel="Northing [m]",
    title="Soil Mg Concentration [ppm]",
)
viz!(ax_mg, grid, color=:white, showfacets=true)

s_mg = Mke.scatter!(
    ax_mg,
    x,
    y,
    color=1e6 .* mg_concentration |> vec, # Scaled to ppm 
    colormap=:turbo,
    markersize=size,
)
Mke.Colorbar(fig_mg[1, 2], s_mg)

Mke.save(joinpath(plotdir, "soil_mg_concentration.png"), fig_mg)
fig_mg

In [ ]:
using CSV, DataFrames, Statistics

csv_files = [joinpath(csvdir, string(Sample_Name, "_$i.csv")) for i in 1:nsim]
df_list = [CSV.read(f, DataFrame) for f in csv_files]
combined_df = vcat(df_list...)

value_cols = setdiff(propertynames(combined_df), [:round, :control])
data_avg = DataFrames.combine(groupby(combined_df, [:round, :control]), value_cols .=> mean .=> value_cols)

println("Ensemble aggregation complete. Total aggregated rows: ", nrow(data_avg))

Mean Soil Mineral X Concentration over Time [ppm]

In [ ]:
fig_min = Mke.Figure(size=(700, 500))

for (idx, min_name) in enumerate(minerals[1:min(2, length(minerals))])
    ax = Mke.Axis(
        fig_min[idx, 1], 
        title="Mean Soil $min_name Concentration [ppm]", 
        xlabel="Round [Years]", 
        ylabel="Concentration [ppm]"
    )
    
    df_treat = filter(row -> row.control == 0, data_avg)
    df_ctrl = filter(row -> row.control == 1, data_avg)
    
    # Treatment line & points
    Mke.lines!(ax, df_treat.round, 1e6 .* df_treat[!, min_name], label="Treatment", color=:dodgerblue, linewidth=2)
    Mke.scatter!(ax, df_treat.round, 1e6 .* df_treat[!, min_name], color=:dodgerblue)
    
    # Control line & points
    Mke.lines!(ax, df_ctrl.round, 1e6 .* df_ctrl[!, min_name], label="Control", color=:orangered, linewidth=2, linestyle=:dash)
    Mke.scatter!(ax, df_ctrl.round, 1e6 .* df_ctrl[!, min_name], color=:orangered)
    
    if idx == 1
        Mke.axislegend(ax, position=:rt)
    end
end

Mke.save(joinpath(plotdir, "mineral_trends.png"), fig_min)
fig_min

Element Concentration over Time [ppm]

In [ ]:
elements = [:Ca, :Mg, :Na, :K]
element_colors = [:firebrick, :forestgreen, :dodgerblue, :darkorchid]

fig_elem = Mke.Figure(size=(700, 900))

for (i, elem) in enumerate(elements)
    ax = Mke.Axis(
        fig_elem[i, 1], 
        title="Soil $elem Concentration [ppm]", 
        xlabel="Round [Years]", 
        ylabel="$elem [ppm]"
    )
    
    # Extract element fraction index (1: Ca, 2: Mg, 3: Na, 4: K)
    df_treat = filter(r -> r.control == 0, data_avg)
    df_ctrl = filter(r -> r.control == 1, data_avg)
    
    treat_vals = map(eachrow(df_treat)) do row
        sum(row[m] * m_element_arr[j][i] for (j, m) in enumerate(minerals))
    end
    
    ctrl_vals = map(eachrow(df_ctrl)) do row
        sum(row[m] * m_element_arr[j][i] for (j, m) in enumerate(minerals))
    end
    
    Mke.lines!(ax, df_treat.round, 1e6 .* treat_vals, label="Treatment", color=element_colors[i], linewidth=2)
    Mke.scatter!(ax, df_treat.round, 1e6 .* treat_vals, color=element_colors[i])
    
    Mke.lines!(ax, df_ctrl.round, 1e6 .* ctrl_vals, label="Control", color=:gray, linewidth=2, linestyle=:dash)
    
    if i == 1
        Mke.axislegend(ax, position=:rt)
    end
end

Mke.save(joinpath(plotdir, "elemental_trends.png"), fig_elem)
fig_elem

Synthetic Deployment Data [ppm]

In [ ]:
using CairoMakie, Random, Statistics

fig_strip = Mke.Figure(size = (650, 500))

ax1 = Mke.Axis(
    fig_strip[1, 1], 
    title = "Synthetic Deployment Data", 
    ylabel = "$(minerals[1]) [ppm]"
)
ax2 = Mke.Axis(
    fig_strip[2, 1], 
    xlabel = "Round", 
    ylabel = "$(minerals[2]) [ppm]"
)

ax1.xticklabelsvisible = false
Mke.linkxaxes!(ax1, ax2)

unique_rounds = sort(unique(combined_df.round))
round_to_idx = Dict(r => i for (i, r) in enumerate(unique_rounds))

control_groups = [
    (val = 0, label = "Treatment", color = :dodgerblue, offset = -0.15),
    (val = 1, label = "Control",   color = :orangered,  offset =  0.15)
]

for (ax_idx, min_name) in enumerate([minerals[1], minerals[2]])
    ax = ax_idx == 1 ? ax1 : ax2
    
    for group in control_groups
        sub_df = filter(row -> row.control == group.val, combined_df)
        
        x_vals = [round_to_idx[r] + group.offset for r in sub_df.round]
        jittered_x = x_vals .+ randn(rng, length(x_vals)) .* 0.04
        y_vals = 1e6 .* sub_df[!, min_name]
        
        Mke.scatter!(
            ax, 
            jittered_x, 
            y_vals, 
            color = group.color, 
            markersize = 5, 
            alpha = 0.5, 
            label = ax_idx == 1 ? group.label : nothing
        )
    end
    
    ax.xticks = (1:length(unique_rounds), string.(unique_rounds))
end

Mke.axislegend(ax1, position = :rt, framevisible = false)

Mke.save(joinpath(plotdir, "stripplot.png"), fig_strip)
fig_strip

CDR Removal Estimate [kg/m^2]

In [ ]:
Ca_i = 2.196
Mg_i = 3.62
Na_i = 1.91
K_i = 1.12
cation_factors = [Ca_i, Mg_i, Na_i, K_i]

sorted_rounds = sort(unique(data_avg.round))
baseline_round = length(sorted_rounds) > 1 ? sorted_rounds[2] : sorted_rounds[1]

cdr_rounds = Float64[]
cdr_values = Float64[]
push!(cdr_rounds, sorted_rounds[1])
push!(cdr_values, 0.0) 

for r_val in sorted_rounds[3:end]
    treat_base = filter(row -> row.control == 0 && row.round == baseline_round, data_avg)
    treat_curr = filter(row -> row.control == 0 && row.round == r_val, data_avg)
    
    if nrow(treat_base) > 0 && nrow(treat_curr) > 0
        loss_sum = 0.0
        for (i, elem) in enumerate([:Ca, :Mg, :Na, :K])
            base_conc = sum(treat_base[1, m] * m_element_arr[j][i] for (j, m) in enumerate(minerals))
            curr_conc = sum(treat_curr[1, m] * m_element_arr[j][i] for (j, m) in enumerate(minerals))
            loss = max(0.0, base_conc - curr_conc)
            loss_sum += loss * cation_factors[i]
        end
        cdr = (loss_sum * 1e6 / A) * d
        push!(cdr_values, cdr)
        push!(cdr_rounds, r_val)
    end
end

fig_cdr = Mke.Figure(size=(600, 400))
ax_cdr = Mke.Axis(
    fig_cdr[1, 1], 
    title="Carbon Dioxide Removal Estimates", 
    xlabel="Round [Years]", 
    ylabel="CDR [kg/m^2]"
)

Mke.lines!(ax_cdr, cdr_rounds, cdr_values, color=:darkgreen, linewidth=2)
Mke.scatter!(ax_cdr, cdr_rounds, cdr_values, color=:darkgreen, markersize=10)

Mke.save(joinpath(plotdir, "CDR_Estimates.png"), fig_cdr)
fig_cdr